In [ ]:
!pip -q install ipywidgets scipy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.optimize import linprog

# ----------------------------------------
# Transportation Cost Optimization System
# ----------------------------------------

def northwest_corner(cost, supply, demand):
    s, d = supply.copy(), demand.copy()
    m, n = cost.shape
    a = np.zeros((m, n), dtype=int)
    i = j = 0
    while i < m and j < n:
        x = min(s[i], d[j])
        a[i, j] = x
        s[i] -= x
        d[j] -= x
        if s[i] == 0 and d[j] == 0:
            i += 1
            j += 1
        elif s[i] == 0:
            i += 1
        else:
            j += 1
    return a

def least_cost(cost, supply, demand):
    s, d = supply.copy(), demand.copy()
    m, n = cost.shape
    a = np.zeros((m, n), dtype=int)
    while sum(s) > 0 and sum(d) > 0:
        cells = [(cost[i, j], i, j)
                 for i in range(m) for j in range(n)
                 if s[i] > 0 and d[j] > 0]
        _, i, j = min(cells)
        x = min(s[i], d[j])
        a[i, j] = x
        s[i] -= x
        d[j] -= x
    return a

def vogel_approximation(cost, supply, demand):
    s, d = supply.copy(), demand.copy()
    m, n = cost.shape
    a = np.zeros((m, n), dtype=int)
    while sum(s) > 0 and sum(d) > 0:
        candidates = []
        for i in range(m):
            if s[i] > 0:
                active = [j for j in range(n) if d[j] > 0]
                vals = sorted(cost[i, j] for j in active)
                penalty = vals[1] - vals[0] if len(vals) > 1 else vals[0]
                j = min(active, key=lambda j: cost[i, j])
                candidates.append((penalty, -cost[i, j], 1, i, j))
        for j in range(n):
            if d[j] > 0:
                active = [i for i in range(m) if s[i] > 0]
                vals = sorted(cost[i, j] for i in active)
                penalty = vals[1] - vals[0] if len(vals) > 1 else vals[0]
                i = min(active, key=lambda i: cost[i, j])
                candidates.append((penalty, -cost[i, j], 0, i, j))
        _, _, _, i, j = max(candidates)
        x = min(s[i], d[j])
        a[i, j] = x
        s[i] -= x
        d[j] -= x
    return a

def stepping_stone_method(cost, supply, demand):
    allocation = vogel_approximation(cost, supply, demand).astype(float)
    m, n = cost.shape

    def get_closed_loop(start_cell, basic_cells):
        def find_path(curr, target, path, find_row):
            if curr == target and len(path) > 3:
                return path
            r, c = curr
            if find_row:
                candidates = [nxt for nxt in basic_cells if nxt[0] == r and nxt != curr]
            else:
                candidates = [nxt for nxt in basic_cells if nxt[1] == c and nxt != curr]
            for nxt in candidates:
                if nxt not in path or (nxt == target and len(path) > 2):
                    result_path = find_path(nxt, target, path + [nxt], not find_row)
                    if result_path is not None:
                        return result_path
            return None

        return find_path(start_cell, start_cell, [start_cell], True)

    while True:
        basic_cells = list(zip(*np.where(allocation > 0)))
        while len(basic_cells) < m + n - 1:
            non_basic = [(i, j) for i in range(m) for j in range(n) if (i, j) not in basic_cells]
            if not non_basic:
                break
            basic_cells.append(non_basic[0])

        u = [None] * m
        v = [None] * n
        u[0] = 0
        solved = False
        while not solved:
            solved = True
            for (i, j) in basic_cells:
                if u[i] is not None and v[j] is None:
                    v[j] = cost[i, j] - u[i]
                    solved = False
                elif v[j] is not None and u[i] is None:
                    u[i] = cost[i, j] - v[j]
                    solved = False
            if not solved and any(x is None for x in u + v):
                for idx, val in enumerate(u):
                    if val is None:
                        u[idx] = 0
                        break
                solved = False

        best_improvement = 0
        best_loop = None
        best_cell = None

        for i in range(m):
            for j in range(n):
                if (i, j) not in basic_cells and allocation[i, j] == 0:
                    idx_val = cost[i, j] - (u[i] + v[j])
                    if idx_val < best_improvement:
                        loop = get_closed_loop((i, j), basic_cells)
                        if loop is not None:
                            best_improvement = idx_val
                            best_loop = loop
                            best_cell = (i, j)

        if best_improvement >= 0 or best_loop is None:
            break

        minus_cells = best_loop[1::2]
        theta = min(allocation[r, c] for (r, c) in minus_cells)

        for idx, (r, c) in enumerate(best_loop):
            if idx % 2 == 0:
                allocation[r, c] += theta
            else:
                allocation[r, c] -= theta

    return np.rint(allocation).astype(int)

# ----------------------------------------
# GUI DESIGN
# ----------------------------------------

sources = widgets.IntSlider(value=3, min=2, max=6, description="Warehouses:")
destinations = widgets.IntSlider(value=3, min=2, max=6, description="Shops:")

build_button = widgets.Button(
    description="Create Input Table",
    button_style="primary",
    icon="table"
)

input_area = widgets.Output()
result_area = widgets.Output()

display(widgets.HTML(
    "<h2>Transportation Cost Optimization System</h2>"
    "<p>Specify the warehouse sizes, then enter supply, demand, and unit transportation costs.</p>"
))
display(widgets.HBox([sources, destinations, build_button]))
display(input_area)
display(result_area)

def build_inputs(_):
    with input_area:
        clear_output(wait=True)

        m, n = sources.value, destinations.value

        # Default example values matching prompt requirements
        default_costs = [
            [2, 3, 1],
            [5, 4, 8],
            [5, 6, 8]
        ]
        default_supply = [30, 40, 20]
        default_demand = [20, 30, 40]

        cost_fields = []
        for i in range(m):
            row_fields = []
            for j in range(n):
                val = default_costs[i][j] if (i < 3 and j < 3) else 1
                row_fields.append(widgets.BoundedIntText(
                    value=val, min=0, max=100000,
                    layout=widgets.Layout(width="90px")
                ))
            cost_fields.append(row_fields)

        supply_fields = []
        for i in range(m):
            val = default_supply[i] if i < 3 else 30
            supply_fields.append(widgets.BoundedIntText(
                value=val, min=0, max=100000,
                description=f"S{i+1}:",
                layout=widgets.Layout(width="150px")
            ))

        demand_fields = []
        for j in range(n):
            val = default_demand[j] if j < 3 else 30
            demand_fields.append(widgets.BoundedIntText(
                value=val, min=0, max=100000,
                description=f"D{j+1}:",
                layout=widgets.Layout(width="150px")
            ))

        cost_table = widgets.VBox([
            widgets.HBox([
                widgets.HTML(f"<b>Source {i+1}</b>", layout=widgets.Layout(width="100px"))
            ] + cost_fields[i])
            for i in range(m)
        ])

        calculate_button = widgets.Button(
            description="Calculate Best Plan",
            button_style="success",
            icon="calculator",
            layout=widgets.Layout(margin="15px 0px 15px 0px", height="40px", width="200px")
        )

        display(widgets.HTML("<h3>1. Transportation costs per unit</h3>"))
        display(widgets.HBox([
            widgets.HTML("<b>Source / Destination</b>", layout=widgets.Layout(width="100px"))
        ] + [
            widgets.HTML(f"<b>D{j+1}</b>", layout=widgets.Layout(width="90px"))
            for j in range(n)
        ]))
        display(cost_table)

        display(widgets.HTML("<h3>2. Available supply</h3>"))
        display(widgets.VBox(supply_fields))

        display(widgets.HTML("<h3>3. Required demand</h3>"))
        # Changed to widgets.VBox to ensure correct rendering without being clipped inside Colab layouts
        display(widgets.VBox(demand_fields))

        display(calculate_button)

        def calculate(_):
            with result_area:
                clear_output(wait=True)

                cost = np.array([                    [field.value for field in row]
                    for row in cost_fields
                ], dtype=float)

                supply = [field.value for field in supply_fields]
                demand = [field.value for field in demand_fields]

                if sum(supply) != sum(demand):
                    display(widgets.HTML(
                        f"<div style='color: red; font-weight: bold;'>"
                        f"Error: Total supply ({sum(supply)}) must equal total demand ({sum(demand)})."
                        f"</div>"
                    ))
                    return

                methods = [
                    ("Northwest Corner", northwest_corner),
                    ("Least Cost", least_cost),
                    ("Vogel's Approximation (VAM)", vogel_approximation),
                    ("Stepping Stone Method (Optimality Tested)", stepping_stone_method)
                ]

                results = []
                for name, method in methods:
                    allocation = method(cost, supply, demand)
                    total = float(np.sum(allocation * cost))
                    results.append((name, allocation, total))

                print("=" * 55)
                print("          TRANSPORTATION PLAN COMPARISON")
                print("=" * 55)

                for name, allocation, total in results:
                    print(f"\n{name.upper()}")
                    print("-" * len(name))
                    df_alloc = pd.DataFrame(
                        allocation,
                        index=[f"S{i+1}" for i in range(len(supply))],
                        columns=[f"D{j+1}" for j in range(len(demand))]
                    )
                    print(df_alloc.to_string())
                    print(f"Total transportation cost: {total:g}")

                best = min(results, key=lambda x: x[2])
                print("\n" + "=" * 55)
                print(f"BEST STRATEGY: {best[0]}")
                print(f"Minimum Transportation Cost: {best[2]:g}")
                print("=" * 55)

                print("\nROUTE-WISE DELIVERY PLAN:")
                for i in range(len(supply)):
                    for j in range(len(demand)):
                        if best[1][i, j] > 0:
                            print(f"  • Source {i+1} -> Destination {j+1}: {best[1][i, j]} units (Cost: {best[1][i, j] * cost[i, j]:g})")

        calculate_button.on_click(calculate)

build_button.on_click(build_inputs)
build_inputs(None)


HTML(value='<h2>Transportation Cost Optimization System</h2><p>Specify the warehouse sizes, then enter supply,…

Output()

Output()